## Libraries 

In [ ]:

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl.worksheet._reader")
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from math import pi

## Dataset

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32

file_path = '1-s2.0-S0022519315005676-mmc1.xlsx'

excel_data = pd.ExcelFile(file_path)
sheet_names = excel_data.sheet_names

raw_x_list, raw_y_list, raw_t_list = [], [], []

for sheet in sheet_names:
    try:
        t_hours = float(sheet.rstrip('h')) 
    except ValueError:
        continue
    data = pd.read_excel(file_path, sheet_name=sheet, header=None)
    x_row = data.iloc[2, 1:].values.astype(np.float64)
    y_row = data.iloc[10, 1:].values.astype(np.float64)

    raw_x_list.extend(x_row.tolist())
    raw_y_list.extend(y_row.tolist())
    raw_t_list.extend([t_hours] * len(x_row))

raw_x = np.array(raw_x_list, dtype=np.float64)
raw_y = np.array(raw_y_list, dtype=np.float64)
raw_t = np.array(raw_t_list, dtype=np.float64) / 24.0

assert raw_x.size == raw_y.size == raw_t.size, "Mismatch"

x_unique_phys = np.unique(raw_x)
t_unique_phys = np.unique(raw_t)

Nx = x_unique_phys.size
Nt = t_unique_phys.size

# Normalization
x_min, x_max = x_unique_phys.min(), x_unique_phys.max()
t_min, t_max = t_unique_phys.min() if Nt > 0 else 0.0, t_unique_phys.max() if Nt > 0 else 1.0

x_span = float(x_max - x_min) if (x_max - x_min) != 0 else 1.0
t_span = float(t_max - t_min) if (t_max - t_min) != 0 else 1.0

def norm_x_phys_to_model(x_phys):
    return (x_phys - x_min) / x_span

def norm_t_phys_to_model(t_phys):
    return (t_phys - t_min) / t_span

# convert to tensors
data_x_phys = torch.tensor(raw_x.reshape(-1,1), dtype=dtype, device=device)   
data_t_phys = torch.tensor(raw_t.reshape(-1,1), dtype=dtype, device=device)   
data_y_phys = torch.tensor(raw_y.reshape(-1,1), dtype=dtype, device=device)   

# normalization
data_x_norm = torch.tensor(norm_x_phys_to_model(raw_x).reshape(-1,1), dtype=dtype, device=device)
data_t_norm = torch.tensor(norm_t_phys_to_model(raw_t).reshape(-1,1), dtype=dtype, device=device)

# physical grid tensors
x_unique_phys_tensor = torch.tensor(x_unique_phys.reshape(-1,1), dtype=dtype, device=device)
t_unique_phys_tensor = torch.tensor(t_unique_phys.reshape(-1,1), dtype=dtype, device=device)

# normalized unique grids
x_unique_norm = torch.tensor(norm_x_phys_to_model(x_unique_phys).reshape(-1,1), dtype=dtype, device=device)
t_unique_norm = torch.tensor(norm_t_phys_to_model(t_unique_phys).reshape(-1,1), dtype=dtype, device=device)

# scale
u_max = float(data_y_phys.max().item()) if data_y_phys.numel()>0 else 1.0
if u_max == 0.0:
    u_max = 1.0

## Fractional

In [ ]:
# Fractional operator
def float_to_tensor(x, device=device, dtype=dtype):
    return torch.tensor(float(x), dtype=dtype, device=device)

def Lgamma_L1(u_mat, gamma, t_grid, lambda_t=None):
    if not isinstance(u_mat, torch.Tensor):
        u_mat = torch.tensor(u_mat, dtype=dtype, device=device)
    if not isinstance(t_grid, torch.Tensor):
        t_grid = torch.tensor(t_grid, dtype=dtype, device=u_mat.device)

    Nx_loc, Nt1 = u_mat.shape
    device_ = u_mat.device
    dtype_ = u_mat.dtype

    if Nt1 > 1:
        diffs = (t_grid[1:] - t_grid[:-1]).abs()
        positive = diffs[diffs > 1e-12]
        if positive.numel() > 0:
            dt_global = float(positive.min().item())
        else:
            dt_global = float((t_grid[1] - t_grid[0]).item()) if Nt1>1 else 1.0
            if dt_global == 0.0:
                dt_global = 1.0
    else:
        dt_global = 1.0

    try:
        lambda_t_val = float(1.0 / dt_global)
    except Exception:
        lambda_t_val = 1.0

    Lg = torch.zeros_like(u_mat, device=device_, dtype=dtype_)
    gamma_t = torch.as_tensor(gamma, dtype=dtype_, device=device_)
    lgamma_term = torch.lgamma(float_to_tensor(2.0, device=device_) - gamma_t)

    for n in range(1, Nt1):
        t_n = float(t_grid[n].item())
        m = int(np.ceil(lambda_t_val * t_n)) if t_n > 0 else 1
        if m < 1:
            m = 1
        dt_local = t_n / m if t_n > 0 else dt_global
        denom = lgamma_term * torch.pow(float_to_tensor(dt_local, device=device_), gamma_t)

        l_idx = torch.arange(0, m, dtype=dtype_, device=device_)
        c_l = torch.pow(l_idx + 1.0, 1.0 - gamma_t) - torch.pow(l_idx, 1.0 - gamma_t)

        aux_times = (torch.arange(0, m, dtype=dtype_, device=device_) * dt_local).to(device_)
        diff = torch.abs(aux_times.unsqueeze(1) - t_grid.unsqueeze(0))
        aux_idx = torch.argmin(diff, dim=1).cpu().numpy().tolist()

        coef0 = -c_l[m - 1]
        coef_current = c_l[0]

        coefs_aux = []
        for k in range(1, m):
            idx1 = m - k
            idx2 = m - k - 1
            coefs_aux.append((c_l[idx1] - c_l[idx2]))

        term = coef0.unsqueeze(0) * u_mat[:, 0]
        term = term + coef_current.unsqueeze(0) * u_mat[:, n]
        for k, ai in enumerate(aux_idx[1:], start=1):
            term = term + coefs_aux[k - 1].unsqueeze(0) * u_mat[:, ai]

        Lg[:, n] = term / denom
    return Lg

def Lalpha_GL_1D(u_mat, alpha, dx, p=1, gl_order=1, out_of_bounds='clamp'):
    if not isinstance(u_mat, torch.Tensor):
        u_mat = torch.tensor(u_mat, dtype=dtype, device=device)
    Nx_loc, Nt1 = u_mat.shape
    device_ = u_mat.device
    dtype_ = u_mat.dtype

    alpha_t = alpha.to(dtype_)
    La = torch.zeros_like(u_mat, device=device_, dtype=dtype_)

    k_max = Nx_loc + 5
    k_idx = torch.arange(0, k_max, dtype=dtype_, device=device_)

    binom = torch.exp(torch.lgamma(alpha_t + 1.0) - torch.lgamma(k_idx + 1.0) - torch.lgamma(alpha_t - k_idx + 1.0))

    def build_delta_for_p(pval):
        vals = []
        for j in range(Nx_loc):
            left_k = j + 1
            right_k = Nx_loc - j
            ks_left = torch.arange(0, left_k, dtype=torch.long, device=device_)
            ks_right = torch.arange(0, right_k, dtype=torch.long, device=device_)
            coeffs_left = ((-1.0) ** ks_left.to(dtype_)) * binom[ks_left]
            coeffs_right = ((-1.0) ** ks_right.to(dtype_)) * binom[ks_right]

            left_idx = (j - (ks_left - pval)).long()
            right_idx = (j + (ks_right - pval)).long()

            if out_of_bounds == 'clamp':
                left_idx = torch.clamp(left_idx, 0, Nx_loc - 1)
                right_idx = torch.clamp(right_idx, 0, Nx_loc - 1)
                u_left = u_mat[left_idx, :]
                u_right = u_mat[right_idx, :]
            elif out_of_bounds == 'zero':
                u_left = torch.zeros((left_k, Nt1), device=device_, dtype=dtype_)
                u_right = torch.zeros((right_k, Nt1), device=device_, dtype=dtype_)
                for ii, idxv in enumerate(left_idx):
                    if 0 <= idxv < Nx_loc:
                        u_left[ii, :] = u_mat[idxv, :]
                for ii, idxv in enumerate(right_idx):
                    if 0 <= idxv < Nx_loc:
                        u_right[ii, :] = u_mat[idxv, :]
            else:
                raise ValueError("Must be 'clamp' or 'zero'")

            left_sum = torch.matmul(coeffs_left.to(dtype_), u_left)
            right_sum = torch.matmul(coeffs_right.to(dtype_), u_right)
            vals.append((left_sum + right_sum) / torch.pow(float_to_tensor(dx, device=device_), alpha_t))
        return torch.stack(vals, dim=0)  

    beta = 1.0 - alpha_t / 2.0
    if gl_order == 1:
        delta_p = build_delta_for_p(p)
        La = delta_p
    elif gl_order == 2:
        delta_p1 = build_delta_for_p(1)
        delta_p0 = build_delta_for_p(0)
        La = (1.0 - beta) * delta_p1 + beta * delta_p0
    elif gl_order == 3:
        A = (11.0 - 6.0 * beta) * (1.0 - beta) / 12.0
        B = (-6.0 * beta * beta + 11.0 * beta + 1.0) / 6.0
        C = (6.0 * beta + 1.0) * (beta - 1.0) / 12.0
        delta_p1 = build_delta_for_p(1)
        delta_p0 = build_delta_for_p(0)
        delta_pm1 = build_delta_for_p(-1)
        La = A * delta_p1 + B * delta_p0 + C * delta_pm1
    else:
        raise ValueError("gl_order must be 1,2 or 3")

    pref = 1.0 / (2.0 * torch.cos(pi * alpha_t / 2.0))
    La = pref * La
    return La

## Parameter Estimation

In [ ]:
class ParameterEstimator(nn.Module):
    def __init__(self):
        super().__init__()
        self.logD0 = nn.Parameter(torch.tensor(np.log(0.01), dtype=dtype, device=device))
        self.logr = nn.Parameter(torch.tensor(np.log(1.68), dtype=dtype, device=device))
        self.logK = nn.Parameter(torch.tensor(np.log(1.7), dtype=dtype, device=device))
        self.raw_alpha = nn.Parameter(torch.tensor(0.5, dtype=dtype, device=device))   
        self.raw_gamma = nn.Parameter(torch.tensor(0.5, dtype=dtype, device=device))   

    def D0(self):
        return torch.exp(self.logD0)

    def r(self):
        return torch.exp(self.logr)

    def K(self):
        return torch.exp(self.logK)

    def alpha(self):
        return 0.02 + 1.98 * torch.sigmoid(self.raw_alpha)

    def gamma(self):
        return 0.01 + 0.99 * torch.sigmoid(self.raw_gamma)

param_estimator = ParameterEstimator().to(device)

# PINN
class PINN(nn.Module):
    def __init__(self, hidden_dim=64):
        super(PINN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x, t):
        x_in = x.to(device).float()
        t_in = t.to(device).float()
        inp = torch.cat([x_in, t_in], dim=1)
        return self.net(inp)

pinn = PINN().to(device)

def predict_on_grid(pinn, x_unique_norm, t_unique_norm):
    Nx_g = x_unique_norm.shape[0]
    Nt_g = t_unique_norm.shape[0]
    X_rep = x_unique_norm.repeat(1, Nt_g).view(-1,1)
    T_rep = t_unique_norm.t().repeat(Nx_g,1).view(-1,1)
    with torch.no_grad():
        u_flat = pinn(X_rep, T_rep)
    u_grid = u_flat.view(Nx_g, Nt_g)
    return u_grid


## ARUS PINN

In [ ]:
def compute_residual_and_uncertainty(pinn, param_estimator):
    pinn.eval()
    with torch.no_grad():
        u_grid_norm = predict_on_grid(pinn, x_unique_norm, t_unique_norm)  # (Nx,Nt)
    u_grid_phys = u_grid_norm * float_to_tensor(u_max, device=device)

    gamma = param_estimator.gamma()
    alpha = param_estimator.alpha()

    if Nx > 1:
        dx_phys = float(x_unique_phys[1] - x_unique_phys[0])
    else:
        dx_phys = 1.0

    Lgamma = Lgamma_L1(u_grid_phys, gamma, t_unique_phys_tensor.squeeze(1))  # (Nx,Nt)
    Lalpha = Lalpha_GL_1D(u_grid_phys, alpha, dx_phys, p=1, gl_order=1, out_of_bounds='clamp')

    D0_val = param_estimator.D0()
    r_val = param_estimator.r()
    K_val = param_estimator.K()

    residual = Lgamma + D0_val * Lalpha - r_val * u_grid_phys * (1.0 - u_grid_phys / K_val)  # (Nx,Nt)

    # Residual absolute
    residual_abs = torch.abs(residual).detach().cpu().numpy().flatten()

    # Uncertainty measure
    uncertainty_tensor = torch.abs(Lgamma) + torch.abs(Lalpha)  # (Nx,Nt)
    uncertainty = uncertainty_tensor.detach().cpu().numpy().flatten()
    return residual_abs, uncertainty, residual.detach()

def indices_from_flat_idx(flat_idx, Nx, Nt):
    i = flat_idx // Nt
    j = flat_idx % Nt
    return int(i), int(j)

def compute_loss_fractional_with_mask(pinn, param_estimator, data_x_norm, data_t_norm, data_x_phys, data_t_phys, data_y_phys, colloc_mask):
    u_grid_norm = predict_on_grid(pinn, x_unique_norm, t_unique_norm)
    u_grid_phys = u_grid_norm * float_to_tensor(u_max, device=device)

    gamma = param_estimator.gamma()
    alpha = param_estimator.alpha()

    if Nx > 1:
        dx_phys = float(x_unique_phys[1] - x_unique_phys[0])
    else:
        dx_phys = 1.0

    Lgamma = Lgamma_L1(u_grid_phys, gamma, t_unique_phys_tensor.squeeze(1))
    Lalpha = Lalpha_GL_1D(u_grid_phys, alpha, dx_phys, p=1, gl_order=1, out_of_bounds='clamp')

    D0_val = param_estimator.D0()
    r_val = param_estimator.r()
    K_val = param_estimator.K()

    residual = Lgamma + D0_val * Lalpha - r_val * u_grid_phys * (1.0 - u_grid_phys / K_val)  # (Nx,Nt)

    mask = colloc_mask.to(residual.device).to(residual.dtype)
    if mask.sum() < 1:
        # avoid zero-division
        mask = torch.ones_like(mask, device=mask.device, dtype=mask.dtype)

    # PDE loss
    pde_loss = torch.sum((residual ** 2) * mask) / torch.sum(mask)

    # DATA loss
    pinn.eval()
    u_pred_at_data_norm = pinn(data_x_norm, data_t_norm)
    u_pred_at_data_phys = u_pred_at_data_norm * float_to_tensor(u_max, device=device)
    data_loss = torch.mean((u_pred_at_data_phys - data_y_phys) ** 2)

    total_loss = data_loss + pde_loss
    return total_loss, data_loss, pde_loss

initial_fraction = 0.5
total_grid_points = Nx * Nt
N_initial = max(1, int(initial_fraction * total_grid_points))

all_indices = np.arange(total_grid_points)
np.random.seed(42)
np.random.shuffle(all_indices)
initial_idx = all_indices[:N_initial]

colloc_mask = torch.zeros((Nx, Nt), dtype=torch.uint8, device=device)
for idx in initial_idx:
    i, j = indices_from_flat_idx(int(idx), Nx, Nt)
    colloc_mask[i, j] = 1

# ARUS training loop
optimizer = optim.Adam(list(pinn.parameters()) + list(param_estimator.parameters()), lr=1e-3)
arus_iters = 30             
epochs_per_arus_iter = 1000    
N_add = 75
alpha_score = 0.5        
eps_r = 0.1                   
eps_u = 0.1                   
loss_history = []

for arus_it in range(arus_iters):
    print(f" ARUS Iteration {arus_it+1}/{arus_iters}")
    pinn.train()
    for epoch in range(epochs_per_arus_iter):
        optimizer.zero_grad()
        total_loss, data_loss, pde_loss = compute_loss_fractional_with_mask(pinn, param_estimator, data_x_norm, data_t_norm, data_x_phys, data_t_phys, data_y_phys, colloc_mask)
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(list(pinn.parameters()) + list(param_estimator.parameters()), max_norm=5.0)
        optimizer.step()

        loss_history.append(total_loss.item())
        if epoch % 100 == 0 or epoch == epochs_per_arus_iter - 1:
            D_val = param_estimator.D0().item()
            print(f" ARUS-iter {arus_it+1} Epoch {epoch}: Total={total_loss.item():.4e}, Data={data_loss.item():.4e}, PDE={pde_loss.item():.4e}")
    residual_abs, uncertainty, residual_tensor = compute_residual_and_uncertainty(pinn, param_estimator)
    mask_flat = colloc_mask.detach().cpu().numpy().astype(bool).flatten()

    remove_mask = (residual_abs < eps_r) & (uncertainty < eps_u) & (mask_flat)
    # keep at least 1% of grid
    keep_idx_list = np.where(~remove_mask)[0]
    min_keep = max(1, int(0.01 * total_grid_points))
    if keep_idx_list.size < min_keep:
        print("skipping removal step.")
    else:
        new_mask_flat = mask_flat.copy()
        new_mask_flat[remove_mask] = False
        colloc_mask = torch.tensor(new_mask_flat.reshape(Nx, Nt), dtype=torch.uint8, device=device)

    # Normalized residual and uncertainty
    eps_small = 1e-12
    R = residual_abs.copy()
    U = uncertainty.copy()
    Rmax = max(R.max(), eps_small)
    Umax = max(U.max(), eps_small)
    score = alpha_score * (R / Rmax) + (1.0 - alpha_score) * (U / Umax)

    # Exclude already selected points
    candidate_indices = np.where(~colloc_mask.detach().cpu().numpy().astype(bool).flatten())[0]
    if candidate_indices.size == 0:
        print("No candidate points available to add.")
    else:
        # Sort
        candidate_scores = score[candidate_indices]
        top_k = min(N_add, candidate_indices.size)
        top_idx_local = np.argsort(candidate_scores)[-top_k:] 
        top_global_idx = candidate_indices[top_idx_local]

        new_mask_flat = colloc_mask.detach().cpu().numpy().astype(bool).flatten()
        new_mask_flat[top_global_idx] = True
        colloc_mask = torch.tensor(new_mask_flat.reshape(Nx, Nt), dtype=torch.uint8, device=device)

    current_count = int(colloc_mask.sum().item())

# Final training refinement
print("Final refinement training:")
refine_epochs = 1000
for epoch in range(refine_epochs):
    optimizer.zero_grad()
    total_loss, data_loss, pde_loss = compute_loss_fractional_with_mask(pinn, param_estimator, data_x_norm, data_t_norm, data_x_phys, data_t_phys, data_y_phys, colloc_mask)
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(list(pinn.parameters()) + list(param_estimator.parameters()), max_norm=5.0)
    optimizer.step()
    loss_history.append(total_loss.item())
    if epoch % 100 == 0 or epoch == refine_epochs - 1:
        print(f"Refine Epoch {epoch}: Total={total_loss.item():.4e}, Data={data_loss.item():.4e}, PDE={pde_loss.item():.4e}")

# Final evaluation
D0_phys = (1e4/24) * torch.exp(param_estimator.D0()).item()
r_phys = (1.0/24.0) * torch.exp(param_estimator.r()).item()
K_phys = (1e3) * torch.exp(param_estimator.K()).item()

print("Final parameter values:")
print(f"D0 = {D0_phys:.6g}")
print(f"r  = {r_phys:.6g}")
print(f"K  = {K_phys:.6g}")
print(f"alpha = {param_estimator.alpha().item():.6g}")
print(f"gamma = {param_estimator.gamma().item():.6g}")